# Section 1 — Data Preparation and Train–Test Split

**Objective:** One reproducible data source for all sections. Fit only on 2021–2023; reserve 2024–2025 for evaluation. 

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

DATA = Path('research_data')
DATA.mkdir(exist_ok=True)

TECH = ['MSFT', 'AAPL', 'NVDA', 'GOOGL', 'META', 'AMZN', 'AMD', 'INTC', 'ORCL', 'CRM']
ENERGY = ['XOM', 'CVX', 'COP', 'EOG', 'OXY', 'SLB', 'HAL', 'VLO', 'MPC', 'PSX']
TICKERS = TECH + ENERGY

START = '2021-01-01'
END = '2026-01-01'

## 1. Download adjusted stock prices

In [9]:
# Download raw data

raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False, threads=True)
prices = raw['Close'].copy().reindex(columns=TICKERS)
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices = prices.sort_index()

print('Rows:', len(prices), 'Range:', prices.index.min(), prices.index.max())

c:\Users\T. Tang\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\T. Tang\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\T. Tang\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#d

Rows: 1255 Range: 2021-01-04 00:00:00 2025-12-31 00:00:00


## 2. Align returns and split the sample

In [10]:
# Calculate daily log return

log_returns = np.log(prices / prices.shift(1)).replace([np.inf, -np.inf], np.nan)
log_returns = log_returns.dropna(subset=TICKERS)

train = log_returns.loc['2021':'2023'].copy()
test = log_returns.loc['2024':'2025'].copy()

print('Training:', len(train), train.index.min().date(), train.index.max().date())
print('Test:', len(test), test.index.min().date(), test.index.max().date())

Training: 752 2021-01-05 2023-12-29
Test: 502 2024-01-02 2025-12-31


## 3. Save canonical inputs

In [11]:
prices.to_csv(DATA / 'adjusted_prices_2021_2025.csv')
log_returns.to_csv(DATA / 'log_returns_2021_2025.csv')
train.to_csv(DATA / 'train_log_returns_2021_2023.csv')
test.to_csv(DATA / 'test_log_returns_2024_2025.csv')

print('Saved to', DATA.resolve())

spots = []

for ticker in TICKERS:
    s = prices.loc['2021':'2023', ticker].dropna()
    spots.append({'ticker': ticker, 'valuation_date': s.index[-1].strftime('%Y-%m-%d'), 'spot': float(s.iloc[-1])})

pd.DataFrame(spots).to_csv(DATA / 'valuation_spots_2023.csv', index=False)

Saved to E:\Quant Finance Camp\My project\research_data
